# Autoencoders and Variational Autoencoders

Before transformers and diffusion models dominated generative AI, **autoencoders** were among the most important neural architectures for learning compressed representations of data. An autoencoder learns to map input into a lower-dimensional **latent space** and then reconstruct the original input from that compact code. When trained well, the latent space captures meaningful structure — smooth neighborhoods, disentangled factors, and geometry that reflects the underlying data distribution.

**Variational Autoencoders (VAEs)** extend this idea into a fully probabilistic generative framework. Instead of learning a single fixed latent code per input, a VAE learns a distribution over latent variables and optimizes an objective called the **Evidence Lower Bound (ELBO)**. VAEs introduced key ideas — continuous latent spaces, the reparameterization trick, and explicit balance between reconstruction fidelity and regularization — that still influence modern generative systems.

This notebook explains how autoencoders fit into generative AI, the main autoencoder variants and their applications, a practical Python demonstration using scikit-learn, and the theory behind variational autoencoders including ELBO, latent space design, and the reparameterization trick.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

---

## 1. Autoencoders in Generative AI

An **autoencoder** is an unsupervised neural network trained to copy its input to its output through a **bottleneck**. The network is split into two parts:

- **Encoder** $f_\theta$: maps input $x$ to a latent representation $z = f_\theta(x)$
- **Decoder** $g_\phi$: maps latent code $z$ back to a reconstruction $\hat{x} = g_\phi(z)$

Training minimizes a **reconstruction loss** such as mean squared error:

$$\mathcal{L}_{\text{recon}} = \|x - \hat{x}\|^2$$

The bottleneck forces the model to learn a compressed representation. If the latent dimension is smaller than the input dimension, the network cannot simply memorize — it must discover structure.

### 1.1 Why Autoencoders Matter for Generation

A standard autoencoder is primarily a **representation learning** tool: it compresses, denoises, and discovers features. For **generation**, the key question is whether the latent space supports **sampling** — can we pick a point $z$ and decode it into a realistic new example?

Classic autoencoders have a limitation: latent codes are learned only for training inputs. Regions of latent space between encoded points may decode to meaningless outputs. **Variational autoencoders** address this by learning a structured, continuous latent distribution rather than isolated codes.

Autoencoders remain conceptually important because they connect three pillars of generative modeling:

1. **Compression** — learning low-dimensional structure in high-dimensional data
2. **Reconstruction** — learning decoders that map abstract codes back to data space
3. **Latent geometry** — organizing data so that nearby latent points correspond to similar outputs

Modern systems such as diffusion models and large language models use different architectures, but the autoencoder intuition — encode, operate in latent space, decode — appears throughout generative pipelines. Image diffusion models often operate in a VAE-compressed latent space rather than raw pixels, reducing compute while preserving perceptual quality.

```
  Input x          Encoder              Latent z           Decoder         Reconstruction x̂
  (784-d)    →   f_θ(x)          →    (k-d)         →   g_φ(z)      →      (784-d)
                     │                                  │
                     └──────── bottleneck forces ────────┘
                              compression
```

---

## 2. Autoencoder Types and Applications

Not all autoencoders are designed the same way. The bottleneck constraint, loss function, and regularization determine what the model learns and how it can be used.

### 2.1 Undercomplete Autoencoders

An **undercomplete** autoencoder has a latent dimension **smaller** than the input dimension ($k < d$). The narrow bottleneck forces the network to learn the most salient features needed for reconstruction — edges in images, phoneme patterns in audio, or principal factors in tabular data.

**Applications:** dimensionality reduction (nonlinear alternative to PCA), feature extraction for downstream classifiers, visualization of high-dimensional data in 2D or 3D latent spaces.

### 2.2 Denoising Autoencoders

A **denoising autoencoder** is trained to reconstruct the **original clean input** from a **corrupted version**. Corruption might be additive Gaussian noise, random pixel dropout, or masking. The encoder must learn robust features that survive noise; the decoder must infer missing structure.

**Applications:** image denoising, inpainting, robust feature learning, pretraining representations before fine-tuning on labeled tasks.

The training objective becomes:

$$\mathcal{L} = \|x - g_\phi(f_\theta(\tilde{x}))\|^2$$

where $\tilde{x}$ is the corrupted input and $x$ is the clean target.

### 2.3 Sparse Autoencoders

A **sparse autoencoder** encourages only a small fraction of latent units to be active for any given input. Sparsity can be enforced by:

- Adding an L1 penalty on latent activations
- Using a KL divergence penalty against a target low-activation rate
- Architectural choices such as tied weights or specific activation functions

**Applications:** learning interpretable features (e.g., edge detectors, stroke detectors), reducing redundancy in representations, and — in modern large language model research — decomposing neural activations into human-interpretable features at scale.

### 2.4 Other Variants (Brief)

| Variant | Key Idea | Typical Use |
|---------|----------|-------------|
| **Overcomplete** | Latent dim > input dim + sparsity penalty | Rich feature dictionaries |
| **Contractive** | Penalize sensitivity of latent code to input perturbations | Robust, stable representations |
| **Convolutional** | Encoder/decoder use conv layers | Images, video, spatial data |
| **Sequence** | RNN or transformer encoder-decoder | Time series, text compression |

The choice of variant depends on whether the goal is compression, denoising, interpretability, or generation.

In [ ]:
# Undercomplete MLP autoencoder on handwritten digits (8×8 images, 64 features)
digits = load_digits()
X = digits.data.astype(np.float64) / 16.0  # scale pixel values to [0, 1]
y = digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Bottleneck architecture: 64 → 32 → 8 → 32 → 64
autoencoder = MLPRegressor(
    hidden_layer_sizes=(32, 8, 32),
    activation="relu",
    solver="adam",
    max_iter=400,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
)
autoencoder.fit(X_train, X_train)

X_recon = autoencoder.predict(X_test)
mse = mean_squared_error(X_test, X_recon)
print(f"Test reconstruction MSE: {mse:.6f}")
print(f"Parameters in network: {sum(w.size for w in autoencoder.coefs_) + sum(b.size for b in autoencoder.intercepts_)}")

# Visualize original vs reconstructed digits
fig, axes = plt.subplots(3, 8, figsize=(12, 5))
for i in range(8):
    axes[0, i].imshow(X_test[i].reshape(8, 8), cmap="gray_r")
    axes[0, i].axis("off")
    axes[1, i].imshow(X_recon[i].reshape(8, 8), cmap="gray_r")
    axes[1, i].axis("off")
    diff = np.abs(X_test[i] - X_recon[i]).reshape(8, 8)
    axes[2, i].imshow(diff, cmap="hot")
    axes[2, i].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=11)
axes[1, 0].set_ylabel("Reconstructed", fontsize=11)
axes[2, 0].set_ylabel("|Error|", fontsize=11)
plt.suptitle("Undercomplete Autoencoder: 64-d input → 8-d bottleneck → 64-d output", y=1.02)
plt.tight_layout()
plt.show()

---

## 3. Implementing Autoencoders in Python

The demonstration above uses `MLPRegressor` from scikit-learn as a practical autoencoder. Key implementation choices:

1. **Symmetric bottleneck** — encoder layers shrink toward the latent dimension; decoder layers expand back. The example uses `(32, 8, 32)` hidden units with 64 input/output features.
2. **Target equals input** — the model learns identity mapping through compression, so `fit(X_train, X_train)` is correct.
3. **Scaling** — pixel values scaled to $[0, 1]$ stabilize training and keep reconstruction loss interpretable.
4. **Early stopping** — prevents overfitting when the network has enough capacity to memorize without learning structure.

In production and research, autoencoders are typically built with **PyTorch**, **TensorFlow/Keras**, or **JAX**, which provide greater flexibility: convolutional layers for images, custom loss functions for denoising, and explicit encoder/decoder modules for latent-space manipulation.

A denoising variant of the same demo would:

- Corrupt inputs: `X_noisy = X + 0.1 * np.random.randn(*X.shape)`
- Train on noisy inputs but keep clean targets: `model.fit(X_noisy, X_clean)`
- Evaluate reconstruction quality on held-out clean images

The reconstruction error heatmaps in the code cell reveal where the 8-dimensional bottleneck loses fine detail — typically stroke endings and thin curves — while preserving global digit shape.

In [ ]:
# Denoising autoencoder: reconstruct clean digits from noisy inputs
rng = np.random.default_rng(42)
noise_std = 0.15
X_train_noisy = np.clip(X_train + rng.normal(0, noise_std, X_train.shape), 0, 1)
X_test_noisy = np.clip(X_test + rng.normal(0, noise_std, X_test.shape), 0, 1)

denoiser = MLPRegressor(
    hidden_layer_sizes=(48, 16, 48),
    activation="relu",
    solver="adam",
    max_iter=400,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
)
denoiser.fit(X_train_noisy, X_train)  # noisy input → clean target
X_denoised = denoiser.predict(X_test_noisy)

print(f"Noisy→clean MSE: {mean_squared_error(X_test, X_denoised):.6f}")
print(f"Raw noisy MSE:    {mean_squared_error(X_test, X_test_noisy):.6f}")

fig, axes = plt.subplots(3, 6, figsize=(10, 5))
for i in range(6):
    axes[0, i].imshow(X_test_noisy[i].reshape(8, 8), cmap="gray_r")
    axes[0, i].axis("off")
    axes[1, i].imshow(X_denoised[i].reshape(8, 8), cmap="gray_r")
    axes[1, i].axis("off")
    axes[2, i].imshow(X_test[i].reshape(8, 8), cmap="gray_r")
    axes[2, i].axis("off")

axes[0, 0].set_ylabel("Noisy", fontsize=11)
axes[1, 0].set_ylabel("Denoised", fontsize=11)
axes[2, 0].set_ylabel("Clean", fontsize=11)
plt.suptitle("Denoising Autoencoder", y=1.02)
plt.tight_layout()
plt.show()

---

## 4. From Deterministic Codes to Generative Models

A standard autoencoder maps each input to a **single point** in latent space. This works well for compression and denoising but creates gaps for generation:

- Latent space may contain large **holes** — regions that do not correspond to any training example
- The decoder may produce **blurry or invalid** outputs when fed arbitrary latent vectors
- There is no explicit **probability model** over data; we cannot compute $p(x)$ or sample systematically

**Generative modeling** requires a latent space where sampling is meaningful. **Variational Autoencoders (VAEs)**, introduced by Kingma and Welling (2013), reformulate the autoencoder as approximate inference in a probabilistic latent variable model.

---

## 5. Variational Autoencoders

A VAE assumes data is generated from latent variables $z$ through a process involving unobserved structure:

$$z \sim p(z), \quad x \sim p_\theta(x \mid z)$$

The **prior** $p(z)$ is usually a standard normal $\mathcal{N}(0, I)$. The **decoder** $p_\theta(x \mid z)$ is a neural network that parameterizes the likelihood (often Gaussian with learned mean, or Bernoulli for binary pixels).

The **encoder** does not output a single $z$. Instead, it outputs parameters of an approximate posterior $q_\phi(z \mid x)$, typically a diagonal Gaussian:

$$q_\phi(z \mid x) = \mathcal{N}(z; \mu_\phi(x), \text{diag}(\sigma_\phi^2(x)))$$

This makes the latent representation **probabilistic**: each input maps to a distribution over codes, not one fixed code.

### 5.1 Latent Space in VAEs

The VAE latent space is shaped by two competing forces:

1. **Reconstruction** — encodings must retain enough information for the decoder to rebuild $x$
2. **Regularization** — encodings are pulled toward the prior $\mathcal{N}(0, I)$, encouraging a smooth, continuous space where interpolation and random sampling work

When training succeeds, walking along a straight line between two latent points $z_1$ and $z_2$ and decoding intermediate points produces gradual morphing between outputs — a hallmark of a good generative latent space.

VAE outputs are often **slightly blurry** compared to GANs because the Gaussian likelihood and MSE-style reconstruction favor averaged predictions. Later architectures (VQ-VAE, diffusion models) address quality while retaining latent-space benefits.

---

## 6. The Evidence Lower Bound (ELBO)

Direct maximization of the marginal likelihood $p_\theta(x) = \int p_\theta(x \mid z) p(z)\, dz$ is intractable because of the integral over latent variables. VAEs instead maximize a lower bound on $\log p_\theta(x)$, called the **ELBO**:

$$\mathcal{L}_{\text{ELBO}}(x) = \mathbb{E}_{q_\phi(z \mid x)}[\log p_\theta(x \mid z)] - D_{\text{KL}}\big(q_\phi(z \mid x) \| p(z)\big)$$

The ELBO has two terms with clear roles:

| Term | Name | Role |
|------|------|------|
| $\mathbb{E}_{q_\phi}[\log p_\theta(x \mid z)]$ | **Reconstruction term** | Decoder must rebuild $x$ from samples of $z$ |
| $D_{\text{KL}}(q_\phi \| p)$ | **KL divergence** | Approximate posterior must stay close to prior |

Maximizing ELBO is equivalent to minimizing:

$$\mathcal{L}_{\text{VAE}} = \underbrace{-\mathbb{E}_{q_\phi}[\log p_\theta(x \mid z)]}_{\text{reconstruction loss}} + \underbrace{D_{\text{KL}}(q_\phi(z \mid x) \| p(z))}_{\text{regularization}}$$

For a Gaussian approximate posterior and standard normal prior, the KL divergence has a closed form:

$$D_{\text{KL}} = -\frac{1}{2} \sum_{j=1}^{k} \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

The KL term prevents the encoder from collapsing to isolated points (which would make sampling impossible) and encourages latent dimensions to be used efficiently — each dimension carries information while remaining close to the prior.

---

## 7. The Reparameterization Trick

The reconstruction term requires sampling $z \sim q_\phi(z \mid x)$, but sampling is not differentiable — gradients cannot flow through a stochastic node to update encoder parameters.

The **reparameterization trick** rewrites the sample as a deterministic function of noise:

$$z = \mu_\phi(x) + \sigma_\phi(x) \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

Now randomness is isolated in $\epsilon$, which does not depend on learnable parameters. Gradients flow through $\mu_\phi$ and $\sigma_\phi$ via $z$ into the decoder loss, enabling end-to-end training with standard backpropagation.

```
  x ──→ Encoder ──→ μ, σ ──→ z = μ + σ·ε ──→ Decoder ──→ x̂
                              ↑
                         ε ~ N(0,I)  (fixed noise, not learned)
```

This trick is a foundational technique in stochastic computation graphs and appears beyond VAEs — in Bayesian neural networks, stochastic depth, and any model that must backpropagate through random variables.

In [ ]:
# ELBO components and reparameterization trick (toy 1-D latent demo)

def kl_divergence_gaussian(mu, log_var):
    """KL( N(mu, sigma^2) || N(0, 1) ) for diagonal Gaussian."""
    return -0.5 * np.sum(1 + log_var - mu**2 - np.exp(log_var))

def reparameterize(mu, log_var, rng):
    """z = mu + sigma * epsilon, epsilon ~ N(0, I)."""
    epsilon = rng.normal(size=mu.shape)
    return mu + np.exp(0.5 * log_var) * epsilon

# Simulate one training example: 4-d "data", 2-d latent
x = np.array([0.8, 0.1, 0.5, 0.9])
mu = np.array([1.2, -0.4])       # encoder output
log_var = np.array([-0.5, 0.3])  # encoder output

rng = np.random.default_rng(42)
z_samples = np.array([reparameterize(mu, log_var, rng) for _ in range(5000)])
kl = kl_divergence_gaussian(mu, log_var)

# Toy decoder: linear reconstruction with fixed weights
W = np.array([[0.6, 0.2], [0.1, 0.8], [0.5, 0.3], [0.7, 0.4]])
x_hat = z_samples @ W.T
recon_loss = np.mean((x - x_hat) ** 2)  # MSE as Gaussian log-likelihood proxy
elbo = -recon_loss - kl  # ELBO = E[log p(x|z)] - KL (signs adjusted for minimization form)

print("Encoder outputs:")
print(f"  mu      = {mu}")
print(f"  log_var = {log_var}")
print(f"  KL(q||p) = {kl:.4f}")
print(f"  Reconstruction MSE = {recon_loss:.4f}")
print(f"  ELBO (higher is better) ≈ {-recon_loss - kl:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Latent samples from reparameterization
axes[0].scatter(z_samples[:, 0], z_samples[:, 1], alpha=0.15, s=8)
axes[0].scatter(mu[0], mu[1], color="red", s=80, label="μ(x)", zorder=5)
theta = np.linspace(0, 2 * np.pi, 100)
axes[0].plot(np.cos(theta), np.sin(theta), "k--", lw=1, label="N(0,I) density contour")
axes[0].set_title("Latent samples via reparameterization")
axes[0].set_xlabel("z₁")
axes[0].set_ylabel("z₂")
axes[0].legend(fontsize=8)

# KL grows as posterior drifts from prior
mu_grid = np.linspace(-2, 2, 50)
kl_curve = [-0.5 * (1 + 0 - m**2 - 1) for m in mu_grid]  # log_var=0 → σ=1
axes[1].plot(mu_grid, kl_curve, color="steelblue")
axes[1].axvline(0, color="gray", ls="--", lw=1)
axes[1].set_title("KL vs μ (1-D, σ=1)")
axes[1].set_xlabel("μ")
axes[1].set_ylabel("KL to N(0,1)")

# ELBO trade-off: reconstruction vs KL weight β
betas = np.linspace(0, 2, 40)
total_loss = [recon_loss + b * kl for b in betas]
axes[2].plot(betas, total_loss, color="darkorange")
axes[2].set_title("Total loss = recon + β·KL")
axes[2].set_xlabel("β (KL weight)")
axes[2].set_ylabel("Loss")

plt.tight_layout()
plt.show()

---

## 8. VAEs in the Generative AI Landscape

VAEs were among the first deep generative models to provide **principled probabilistic training** with stable optimization. They influenced later systems in several ways:

- **Stable-GAN and VAE-GAN hybrids** combined VAE structure with adversarial losses for sharper outputs
- **VQ-VAE** replaced continuous latents with learned discrete codebooks, improving image quality and enabling autoregressive priors over codes
- **Latent diffusion** trains diffusion models in a VAE-compressed space (e.g., Stable Diffusion), dramatically reducing compute while keeping perceptual fidelity

Compared to modern alternatives:

| Model | Strengths | Limitations |
|-------|-----------|-------------|
| **VAE** | Principled objective, smooth latent space, stable training | Often blurry outputs |
| **GAN** | Sharp, realistic samples | Training instability, mode collapse |
| **Diffusion** | State-of-the-art image quality | Slow iterative sampling |
| **Autoregressive** | Strong for discrete sequences (text, codes) | Sequential generation latency |

Understanding autoencoders and VAEs provides the conceptual bridge between classical dimensionality reduction and modern generative pipelines — especially the idea that generation can happen in a learned, lower-dimensional latent space rather than directly in raw pixel or token space.

---

## 9. Summary

**Autoencoders** learn to compress data through an encoder-decoder architecture with a bottleneck, minimizing reconstruction error. Undercomplete autoencoders perform nonlinear dimensionality reduction; denoising autoencoders learn robust features from corrupted inputs; sparse autoencoders encourage interpretable, selective activations. They are powerful for representation learning and preprocessing, but standard autoencoders do not define a complete generative model over latent space.

**Variational Autoencoders** place a probabilistic framework around the autoencoder: the encoder outputs a distribution $q_\phi(z \mid x)$, the decoder models $p_\theta(x \mid z)$, and training maximizes the **ELBO** — balancing reconstruction fidelity against a KL penalty that regularizes the latent space toward a standard normal prior. The **reparameterization trick** makes sampling differentiable, enabling end-to-end gradient-based learning.

These ideas remain relevant in generative AI today. Latent-space compression appears in image diffusion systems, sequence models use encoder-decoder structure for transformation tasks, and the ELBO decomposition — reconstruction plus regularization — echoes in many variational and probabilistic training objectives. Autoencoders and VAEs are not the final word in generation, but they are essential foundations for understanding how neural networks learn to encode, reconstruct, and ultimately synthesize data.